# Non-parametric regression with k-nearest neighbors

## Initialize

In [165]:
# import required libraries
import numpy as np
from numpy.polynomial.polynomial import Polynomial
import matplotlib.pyplot as plt
from scipy.optimize import minimize
import tensorflow as tf #type: ignore
import torch #type: ignore
import torch.nn.functional as F #type: ignore
#
from IPython.display import display, Math
# Import typyng library for type hinting
from typing import Callable, Union, List, Any, Tuple

## Numpy/Scipy

### 2NN (polynomial data)

In [65]:
def y(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

def pred_func(x_data: np.ndarray, 
              train_data: np.ndarray
             ) -> np.ndarray:
    """
    Computes the predictions for the given x_data using a k-nearest neighbors approach 
    based on the training data. If x_data is the same as x_train, it uses the 2nd and 3rd 
    nearest neighbors to avoid self-match; otherwise, it uses the two nearest neighbors.
    
    Args:
        x_data (np.ndarray): Array of input x values for which to predict y.
        train_data (np.ndarray): Array of training data with shape (N, 2), where 
                               train_data[:, 0] are the x values and train_data[:, 1] are the y values.
    
    Returns:
        A numpy array of predictions, each row [x, pred_y].
    """
    x_train = train_data[:, 0]
    y_train = train_data[:, 1]
    
    # Determine if x_data has the same shape as the training x values.
    if x_data.shape == x_train.shape:
        # If any element in x_data exactly equals an element in x_train,
        # assume we're predicting on training data.
        if np.any(x_data == x_train):
            # For each x in x_train, use the 2nd and 3rd nearest neighbors.
            pred = np.array([
                [x, np.mean(y_train[np.argsort(np.abs(x_train - x))[1:3]])]
                for x in x_train
            ])
        else:
            # Otherwise, for each x in x_data, use the two nearest neighbors.
            pred = np.array([
                [x, np.mean(y_train[np.argsort(np.abs(x_train - x))[0:2]])]
                for x in x_data
            ])
    else:
        # If shapes differ, treat x_data as new input: use the two nearest neighbors.
        pred = np.array([
            [x, np.mean(y_train[np.argsort(np.abs(x_train - x))[0:2]])]
            for x in x_data
        ])
    
    return pred

def MSE(y_true: np.ndarray, 
        y_pred: np.ndarray
       ) -> float:
    """
    Computes the mean squared error between the true and predicted y values.
    
    Args:
        y_true: A numpy array of true y values.
        y_pred: A numpy array of predicted y values.
    
    Returns:
        The mean squared error between the true and predicted y values.
    """
    return np.mean((y_true - y_pred)**2)

In [66]:
# Generate data coordinates
x_train: np.ndarray = np.arange(-3, 4, 1)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat(x_train)
y_val = y_stat(x_val)
y_true = y(x_true)

train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)
train_data

array([[-3.        ,  2.87523453],
       [-2.        ,  3.00934707],
       [-1.        ,  2.7780358 ],
       [ 0.        ,  1.74756396],
       [ 1.        ,  4.27298745],
       [ 2.        ,  4.51421884],
       [ 3.        ,  1.84617967]])

In [67]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.89369144]
 [-2.5         2.9422908 ]
 [-2.          2.82663516]
 [-1.5         2.89369144]
 [-1.          2.37845552]
 [-0.5         2.26279988]
 [ 0.          3.52551163]
 [ 0.5         3.01027571]
 [ 1.          3.1308914 ]
 [ 1.5         4.39360315]
 [ 2.          3.05958356]
 [ 2.5         3.18019926]
 [ 3.          4.39360315]]


In [68]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


### 2NN (non-polynomial data)

In [69]:
def y(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return 3 + np.sin(x) - np.cos(x)

def y_stat(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

In [70]:
# Generate data coordinates
x_train: np.ndarray = np.arange(-3, 4, 1)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat(x_train)
y_val = y_stat(x_val)
y_true = y(x_true)

train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)
train_data

array([[-3.        ,  2.09910702],
       [-2.        ,  2.84952981],
       [-1.        ,  2.77126251],
       [ 0.        ,  1.74756396],
       [ 1.        ,  4.28248947],
       [ 2.        ,  4.8396631 ],
       [ 3.        ,  4.35229217]])

In [71]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.81039616]
 [-2.5         2.47431841]
 [-2.          2.43518476]
 [-1.5         2.81039616]
 [-1.          2.29854689]
 [-0.5         2.25941324]
 [ 0.          3.52687599]
 [ 0.5         3.01502671]
 [ 1.          3.29361353]
 [ 1.5         4.56107629]
 [ 2.          4.31739082]
 [ 2.5         4.59597764]
 [ 3.          4.56107629]]


In [72]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 0.766, Validation loss: 0.466


### Generalized 2NN (polynomial data)

In [ ]:
def y(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

def pred_func(x_data: np.ndarray,
              train_data: np.ndarray,
              m: float = 1.0,
              n: float = 1.0
             ) -> np.ndarray:
    """
    Computes the predictions for the given x_data using a k-nearest neighbors approach 
    based on the training data. If x_data is the same as x_train, it uses the 2nd and 3rd 
    nearest neighbors to avoid self-match; otherwise, it uses the two nearest neighbors.
    
    Args:
        x_data (np.ndarray): Array of input x values for which to predict y.
        train_data (np.ndarray): Array of training data with shape (N, 2), where 
                               train_data[:, 0] are the x values and train_data[:, 1] are the y values.
        m (float): The m parameter of the modified distance metric.
        n (float): The n parameter of the modified distance metric.                 
    
    Returns:
        A numpy array of predictions, each row [x, pred_y].
    """
    x_train = train_data[:, 0]
    y_train = train_data[:, 1]
    
    # Determine if x_data has the same shape as the training x values.
    if x_data.shape == x_train.shape:
        # If any element in x_data exactly equals an element in x_train,
        # assume we're predicting on training data.
        if np.any(x_data == x_train):
            # For each x in x_train, use the 2nd and 3rd nearest neighbors.
            pred = np.array([
                [x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[1:3]])**n)**(1/m)]
                for x in x_train
            ])
        else:
            # Otherwise, for each x in x_data, use the two nearest neighbors.
            pred = np.array([
                [x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[0:2]])**n)**(1/m)]
                for x in x_data
            ])
    else:
        # If shapes differ, treat x_data as new input: use the two nearest neighbors.
        pred = np.array([
            [x, 1/2*np.sum(np.abs(y_train[np.argsort(np.abs(x_train - x))[0:2]])**n)**(1/m)]
            for x in x_data
        ])
    
    return pred

def loss_func(x_data: np.ndarray,
              y_data: np.ndarray,
              train_data: np.ndarray,
              m: float = 1.0,
              n: float = 1.0
             ) -> float:
    """
    Computes the loss for the given data (x_data, y_data) using the training data as input.
    """
    pred = pred_func(x_data, train_data, m, n)
    return np.mean((y_data - pred[:, 1])**2)

In [97]:
# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 4., 1.)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat(x_train)
y_val = y_stat(x_val)
y_true = y(x_true)

train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)
train_data

array([[-3.        ,  4.45002345],
       [-2.        ,  2.70093471],
       [-1.        ,  1.74030358],
       [ 0.        ,  1.9747564 ],
       [ 1.        ,  3.38979875],
       [ 2.        ,  4.05142188],
       [ 3.        ,  1.64711797]])

In [98]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.22061914]
 [-2.5         3.57547908]
 [-2.          3.09516352]
 [-1.5         2.22061914]
 [-1.          2.33784555]
 [-0.5         1.85752999]
 [ 0.          2.56505116]
 [ 0.5         2.68227757]
 [ 1.          3.01308914]
 [ 1.5         3.72061031]
 [ 2.          2.51845836]
 [ 2.5         2.84926993]
 [ 3.          3.72061031]]


In [80]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


In [81]:
train_loss = loss_func(x_train, y_train, train_data, 1.0, 1.0)
val_loss = loss_func(x_val, y_val, train_data, 1.0, 1.0)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


#### Optimizing loss function over train data

In [82]:
def objective_train(params):
    # Here, params = [m, n]
    m, n = params
    return loss_func(x_train, y_train, train_data, m, n)
print(objective_train([1.0, 1.0]))
print(objective_train([0.5, 2.0]))
print(objective_train([2.0, 0.5]))

1.8948857380468203
118937.15251614539
5.310191258361994


In [83]:
# Initial guess for the parameters [m, n]:
initial_params = [1.0, 1.0]

# Optimize on the training data:
result_train = minimize(objective_train, initial_params, method='Powell', options={'xtol': 1e-8, 'ftol': 1e-8})
optimal_params = result_train.x
train_loss_opt = loss_func(x_train, y_train, train_data, *optimal_params)
print("Optimal parameters on training set:", optimal_params)
print("Train loss:", train_loss_opt)

# Now, evaluate the validation loss using these optimal parameters:
val_loss_opt = loss_func(x_val, y_val, train_data, *optimal_params)
print("Validation loss:", val_loss_opt)

Optimal parameters on training set: [ 0.24395654 -0.23740719]
Train loss: 0.6813074745247688
Validation loss: 2.123088412770937


#### Optimizing loss function over validation data

In [84]:
def objective_val(params):
    # Here, params = [m, n]
    m, n = params
    return loss_func(x_val, y_val, train_data, m, n)
print(objective_val([1.0, 1.0]))
print(objective_val([0.5, 2.0]))
print(objective_val([2.0, 0.5]))

0.40437371232142216
120663.6629126066
3.343110026002987


In [85]:
# Initial guess for the parameters [m, n]:
initial_params = [1.0, 1.0]

# Optimize on the training data:
result_val = minimize(objective_val, initial_params, method='Powell', options={'xtol': 1e-8, 'ftol': 1e-8})
optimal_params = result_val.x
train_loss_opt = loss_func(x_train, y_train, train_data, *optimal_params)
print("Optimal parameters on training set:", optimal_params)
print("Train loss:", train_loss_opt)

# Now, evaluate the validation loss using these optimal parameters:
val_loss_opt = loss_func(x_val, y_val, train_data, *optimal_params)
print("Validation loss:", val_loss_opt)

Optimal parameters on training set: [1.64756999 1.76917139]
Train loss: 1.7087540257863292
Validation loss: 0.1578040758316644


### Generalized 2NN (polynomial data) - Large Sample

In [104]:
def y(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 0.1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 0.1)
    
    return noisy_y_values

In [105]:
# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 3.1, 0.1)
x_val: np.ndarray = np.arange(-2.95, 2.75, 0.2)
x_test: np.ndarray = np.arange(-2.85, 2.85, 0.2)
x_true: np.ndarray = np.arange(-3., 3.1, 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat(x_train)
y_val = y_stat(x_val)
y_test = y_stat(x_test)
y_true = y(x_true)

train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
test_data = np.concatenate((x_test.reshape(-1, 1), y_test.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

In [106]:
print(x_train[0:10])
print(x_val[0:10])
print(x_test[0:10])
print(x_train[0:10]-x_val[0:10])
print(x_train[0:10]-x_test[0:10])
print(x_val[0:10]-x_test[0:10])

[-3.  -2.9 -2.8 -2.7 -2.6 -2.5 -2.4 -2.3 -2.2 -2.1]
[-2.95 -2.75 -2.55 -2.35 -2.15 -1.95 -1.75 -1.55 -1.35 -1.15]
[-2.85 -2.65 -2.45 -2.25 -2.05 -1.85 -1.65 -1.45 -1.25 -1.05]
[-0.05 -0.15 -0.25 -0.35 -0.45 -0.55 -0.65 -0.75 -0.85 -0.95]
[-0.15 -0.25 -0.35 -0.45 -0.55 -0.65 -0.75 -0.85 -0.95 -1.05]
[-0.1 -0.1 -0.1 -0.1 -0.1 -0.1 -0.1 -0.1 -0.1 -0.1]


In [107]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
pred_test = pred_func(x_test, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]

In [108]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
test_loss = MSE(y_test, pred_test[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}, Test loss: {test_loss:.3f}")

Train loss: 0.021, Validation loss: 0.011, Test loss: 0.021


In [109]:
train_loss = loss_func(x_train, y_train, train_data, 1.0, 1.0)
val_loss = loss_func(x_val, y_val, train_data, 1.0, 1.0)
test_loss = loss_func(x_test, y_test, train_data, 1.0, 1.0)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}, Test loss: {test_loss:.3f}")

Train loss: 0.021, Validation loss: 0.011, Test loss: 0.021


#### Optimizing loss function over train data

In [110]:
def objective_train(params):
    # Here, params = [m, n]
    m, n = params
    return loss_func(x_train, y_train, train_data, m, n)
print(objective_train([1.0, 1.0]))
print(objective_train([0.99, 0.99]))
print(objective_train([1.1, 1.1]))
print(objective_train([0.9, 1.1]))
print(objective_train([1.1, 0.9]))

0.020867268685917803
0.021405212086726827
0.05233510375686124
1.4930417376261915
0.5353353627705292


In [111]:
# Initial guess for the parameters [m, n]:
initial_params = [1.0, 1.0]

# Optimize on the training data:
result_train = minimize(objective_train, initial_params, method='Nelder-Mead', options={'xtol': 1e-8, 'ftol': 1e-8})
optimal_params = result_train.x
train_loss_opt = loss_func(x_train, y_train, train_data, *optimal_params)
print("Optimal parameters on training set:", optimal_params)
print("Train loss:", train_loss_opt)

# Now, evaluate the validation loss using these optimal parameters:
val_loss_opt = loss_func(x_val, y_val, train_data, *optimal_params)
print("Validation loss:", val_loss_opt)

# Now, evaluate the test loss using these optimal parameters:
test_loss_opt = loss_func(x_test, y_test, train_data, *optimal_params)
print("Test loss:", test_loss_opt)

Optimal parameters on training set: [1.03507632 1.05502008]
Train loss: 0.020633723279625395
Validation loss: 0.010612547148087608
Test loss: 0.01961604621461617


/var/folders/8p/lb_zz6bs7g7cdkt7y_yjy0z40000gn/T/ipykernel_38518/3970587528.py:5: OptimizeWarning: Unknown solver options: xtol, ftol
  result_train = minimize(objective_train, initial_params, method='Nelder-Mead', options={'xtol': 1e-8, 'ftol': 1e-8})


#### Optimizing loss function over validation data

In [112]:
def objective_val(params):
    # Here, params = [m, n]
    m, n = params
    return loss_func(x_val, y_val, train_data, m, n)
print(objective_val([1.0, 1.0]))
print(objective_val([0.99, 0.99]))
print(objective_val([1.1, 1.1]))
print(objective_val([0.9, 1.1]))
print(objective_val([1.1, 0.9]))

0.011172250934924315
0.011402588044857349
0.04641712641272331
1.5276135052922213
0.5609926496973789


In [113]:
# Initial guess for the parameters [m, n]:
initial_params = [1.0, 1.0]

# Optimize on the training data:
result_val = minimize(objective_val, initial_params, method='Powell', options={'xtol': 1e-8, 'ftol': 1e-8})
optimal_params = result_val.x
train_loss_opt = loss_func(x_train, y_train, train_data, *optimal_params)
print("Optimal parameters on training set:", optimal_params)
print("Train loss:", train_loss_opt)

# Now, evaluate the validation loss using these optimal parameters:
val_loss_opt = loss_func(x_val, y_val, train_data, *optimal_params)
print("Validation loss:", val_loss_opt)

# Now, evaluate the test loss using these optimal parameters:
test_loss_opt = loss_func(x_test, y_test, train_data, *optimal_params)
print("Test loss:", test_loss_opt)

Optimal parameters on training set: [1.05843285 1.09415469]
Train loss: 0.02080467704398237
Validation loss: 0.010429915552680028
Test loss: 0.01948174476845858


## TensorFlow2

### 2NN (polynomial data)

In [ ]:
def y(x: tf.Tensor) -> tf.Tensor:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A tf tensor of x values.
    
    Returns:
        A tf tensor of y values computed from the polynomial expression.
    """
    return tf.cast(-tf.pow(x, 4)/24 - tf.pow(x, 3)/6 + tf.pow(x, 2)/2 + x + 2, dtype=tf.float64)
    
def y_stat(x: tf.Tensor) -> tf.Tensor:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A tf tensor of x values.
    
    Returns:
        A tf tensor of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: tf.Tensor = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: tf.Tensor = true_y_values + tf.random.normal(tf.shape(true_y_values), mean=0.0, stddev=1.0, dtype=tf.float64)
    
    return noisy_y_values

def y_numpy(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat_numpy(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y_numpy(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

def pred_func(x_data: tf.Tensor, 
              train_data: tf.Tensor
             ) -> tf.Tensor:
    """
    Computes the predictions for the given x_data using a k-nearest neighbors approach 
    based on the training data. If x_data is the same as x_train, it uses the 2nd and 3rd 
    nearest neighbors (to avoid self-match); otherwise, it uses the two nearest neighbors.
    
    Args:
        x_data (tf.Tensor): Tensor of input x values for which to predict y.
        train_data (tf.Tensor): Tensor of training data with shape (N, 2), where 
                                train_data[:, 0] contains the x values and 
                                train_data[:, 1] contains the corresponding y values.
    
    Returns:
        tf.Tensor: A tensor of predictions, each row is [x, pred_y].
    """
    x_train = train_data[:, 0]
    y_train = train_data[:, 1]
    
    # If x_data has the same shape as x_train and any element is equal, assume we're predicting on training data.
    if x_data.shape == x_train.shape and tf.reduce_any(tf.equal(x_data, x_train)):
        input_tensor = x_train
    else:
        input_tensor = x_data

    def compute_pred_for_x(x):
        # Compute absolute differences between this x and all training x's.
        diffs = tf.abs(x_train - x)
        # Sort indices of training points by increasing difference.
        sorted_indices = tf.argsort(diffs, direction='ASCENDING')
        # If x is in the training set, skip the closest (self-match) and take the 2nd and 3rd nearest.
        # Otherwise, take the two nearest neighbors.
        if tf.reduce_any(tf.equal(x, x_train)):
            neighbor_indices = sorted_indices[1:3]
        else:
            neighbor_indices = sorted_indices[0:2]
        gathered = tf.gather(y_train, neighbor_indices)
        pred_y = tf.reduce_mean(gathered)
        return tf.stack([x, pred_y])
    
    pred = tf.map_fn(compute_pred_for_x, input_tensor, dtype=tf.float64)
    return pred

def MSE(y_true: tf.Tensor, 
        y_pred: tf.Tensor
       ) -> tf.Tensor:
    """
    Calculates the Mean Squared Error (MSE) using tensorflow MSE loss function.
    
    Args:
        x_values: A 1D tf.Tensor of x values for evaluation.
        y_true: A 1D tf.Tensor of true y values.
    
    Returns:
        The MSE as a tf.Tensor.
    """
    loss: tf.Tensor = tf.losses.MSE(y_true, y_pred)
    return loss

In [138]:
# Generate data coordinates using tf.range
x_train = tf.range(-3, 4, 1, dtype=tf.float64)
x_val   = tf.range(-2.5, 3.5, 1, dtype=tf.float64)
x_true  = tf.range(-3.0, 4.0, 0.5, dtype=tf.float64)

# Generate target values through the statistical model
tf.random.set_seed(100)
y_train = y_stat(x_train)
y_val   = y_stat(x_val)
y_true  = y(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = tf.concat([tf.expand_dims(x_train, axis=1), tf.expand_dims(y_train, axis=1)], axis=1)
val_data   = tf.concat([tf.expand_dims(x_val, axis=1), tf.expand_dims(y_val, axis=1)], axis=1)
true_data  = tf.concat([tf.expand_dims(x_true, axis=1), tf.expand_dims(y_true, axis=1)], axis=1)
train_data

# Notice that the instance of data generated by TensorFlow2 is different than the one generated by numpy. To reproduce the same results as with numpy, one would need to generate data with numpy and then convert it to TensorFlow2 tensors. Comment/uncomment the following lines to use different/same data

# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 4., 1.)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat_numpy(x_train)
y_val = y_stat_numpy(x_val)
y_true = y_numpy(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

# Convert numpy arrays to TensorFlow2 tensors
x_train = tf.constant(x_train, dtype=tf.float64)
y_train = tf.constant(y_train, dtype=tf.float64)
x_val = tf.constant(x_val, dtype=tf.float64)
y_val = tf.constant(y_val, dtype=tf.float64)
x_true = tf.constant(x_true, dtype=tf.float64)
y_true = tf.constant(y_true, dtype=tf.float64)
train_data = tf.constant(train_data, dtype=tf.float64)
val_data = tf.constant(val_data, dtype=tf.float64)
true_data = tf.constant(true_data, dtype=tf.float64)
train_data

<tf.Tensor: shape=(7, 2), dtype=float64, numpy=
array([[-3.        ,  2.87523453],
       [-2.        ,  3.00934707],
       [-1.        ,  2.7780358 ],
       [ 0.        ,  1.74756396],
       [ 1.        ,  4.27298745],
       [ 2.        ,  4.51421884],
       [ 3.        ,  1.84617967]])>

In [136]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          1.95942478]
 [-2.5         3.73530946]
 [-2.          2.94321876]
 [-1.5         1.95942478]
 [-1.          2.28946473]
 [-0.5         1.49737403]
 [ 0.          1.90963798]
 [ 0.5         2.23967793]
 [ 1.          3.43872568]
 [ 1.5         3.85098963]
 [ 2.          3.14561242]
 [ 2.5         4.34466017]
 [ 3.          3.85098963]]


In [137]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.887, Validation loss: 1.002


### 2NN (non-polynomial data)

In [139]:
def y(x: tf.Tensor) -> tf.Tensor:
    """Computes the function 3 + sin(x) - cos(x) for given x values.
    
    Args:
        x: A tf tensor of x values.
    
    Returns:
        A tf tensor of y values computed from the function.
    """
    return 3. + tf.sin(x) - tf.cos(x)

def y_stat(x: tf.Tensor) -> tf.Tensor:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A tf tensor of x values.
    
    Returns:
        A tf tensor of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: tf.Tensor = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: tf.Tensor = true_y_values + tf.random.normal(tf.shape(true_y_values), mean=0.0, stddev=1.0, dtype=tf.float64)
    
    return noisy_y_values

def y_numpy(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return 3 + np.sin(x) - np.cos(x)

def y_stat_numpy(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y_numpy(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

In [140]:
# Generate data coordinates using tf.range
x_train = tf.range(-3, 4, 1, dtype=tf.float64)
x_val   = tf.range(-2.5, 3.5, 1, dtype=tf.float64)
x_true  = tf.range(-3.0, 4.0, 0.5, dtype=tf.float64)

# Generate target values through the statistical model
tf.random.set_seed(100)
y_train = y_stat(x_train)
y_val   = y_stat(x_val)
y_true  = y(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = tf.concat([tf.expand_dims(x_train, axis=1), tf.expand_dims(y_train, axis=1)], axis=1)
val_data   = tf.concat([tf.expand_dims(x_val, axis=1), tf.expand_dims(y_val, axis=1)], axis=1)
true_data  = tf.concat([tf.expand_dims(x_true, axis=1), tf.expand_dims(y_true, axis=1)], axis=1)
train_data

# Notice that the instance of data generated by TensorFlow2 is different than the one generated by numpy. To reproduce the same results as with numpy, one would need to generate data with numpy and then convert it to TensorFlow2 tensors. Comment/uncomment the following lines to use different/same data

# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 4., 1.)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat_numpy(x_train)
y_val = y_stat_numpy(x_val)
y_true = y_numpy(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

# Convert numpy arrays to TensorFlow2 tensors
x_train = tf.constant(x_train, dtype=tf.float64)
y_train = tf.constant(y_train, dtype=tf.float64)
x_val = tf.constant(x_val, dtype=tf.float64)
y_val = tf.constant(y_val, dtype=tf.float64)
x_true = tf.constant(x_true, dtype=tf.float64)
y_true = tf.constant(y_true, dtype=tf.float64)
train_data = tf.constant(train_data, dtype=tf.float64)
val_data = tf.constant(val_data, dtype=tf.float64)
true_data = tf.constant(true_data, dtype=tf.float64)
train_data

<tf.Tensor: shape=(7, 2), dtype=float64, numpy=
array([[-3.        ,  2.09910702],
       [-2.        ,  2.84952981],
       [-1.        ,  2.77126251],
       [ 0.        ,  1.74756396],
       [ 1.        ,  4.28248947],
       [ 2.        ,  4.8396631 ],
       [ 3.        ,  4.35229217]])>

In [141]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.81039616]
 [-2.5         2.47431841]
 [-2.          2.43518476]
 [-1.5         2.81039616]
 [-1.          2.29854689]
 [-0.5         2.25941324]
 [ 0.          3.52687599]
 [ 0.5         3.01502671]
 [ 1.          3.29361353]
 [ 1.5         4.56107629]
 [ 2.          4.31739082]
 [ 2.5         4.59597764]
 [ 3.          4.56107629]]


In [142]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 0.766, Validation loss: 0.466


### Generalized 2NN (polynomial data)

In [143]:
def y(x: tf.Tensor) -> tf.Tensor:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A tf tensor of x values.
    
    Returns:
        A tf tensor of y values computed from the polynomial expression.
    """
    return tf.cast(-tf.pow(x, 4)/24 - tf.pow(x, 3)/6 + tf.pow(x, 2)/2 + x + 2, dtype=tf.float64)
    
def y_stat(x: tf.Tensor) -> tf.Tensor:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A tf tensor of x values.
    
    Returns:
        A tf tensor of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: tf.Tensor = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: tf.Tensor = true_y_values + tf.random.normal(tf.shape(true_y_values), mean=0.0, stddev=1.0, dtype=tf.float64)
    
    return noisy_y_values

def y_numpy(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat_numpy(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y_numpy(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

def pred_func(x_data: tf.Tensor, 
              train_data: tf.Tensor,
              m: float = 1.0,
              n: float = 1.0
             ) -> tf.Tensor:
    """
    Computes the predictions for the given x_data using a k-nearest neighbors approach 
    based on the training data. If x_data is the same as x_train, it uses the 2nd and 3rd 
    nearest neighbors (to avoid self-match); otherwise, it uses the two nearest neighbors.
    
    Args:
        x_data (tf.Tensor): Tensor of input x values for which to predict y.
        train_data (tf.Tensor): Tensor of training data with shape (N, 2), where 
                                train_data[:, 0] contains the x values and 
                                train_data[:, 1] contains the corresponding y values.
        m (float): The m parameter of the modified distance metric.
        n (float): The n parameter of the modified distance metric.              
    
    Returns:
        tf.Tensor: A tensor of predictions, each row is [x, pred_y].
    """
    x_train = train_data[:, 0]
    y_train = train_data[:, 1]
    
    # If x_data has the same shape as x_train and any element is equal, assume we're predicting on training data.
    if x_data.shape == x_train.shape and tf.reduce_any(tf.equal(x_data, x_train)):
        input_tensor = x_train
    else:
        input_tensor = x_data

    def compute_pred_for_x(x):
        # Compute absolute differences between this x and all training x's.
        diffs = tf.abs(x_train - x)
        # Sort indices of training points by increasing difference.
        sorted_indices = tf.argsort(diffs, direction='ASCENDING')
        # If x is in the training set, skip the closest (self-match) and take the 2nd and 3rd nearest.
        # Otherwise, take the two nearest neighbors.
        if tf.reduce_any(tf.equal(x, x_train)):
            neighbor_indices = sorted_indices[1:3]
        else:
            neighbor_indices = sorted_indices[0:2]
        gathered = tf.gather(y_train, neighbor_indices)
        pred_y = 1/2*tf.reduce_sum(tf.abs(gathered)**n)**(1/m)
        return tf.stack([x, pred_y])
    
    pred = tf.map_fn(compute_pred_for_x, input_tensor, dtype=tf.float64)
    return pred

def loss_func(x_data: tf.Tensor,
              y_data: tf.Tensor,
              train_data: tf.Tensor,
              m: float = 1.0,
              n: float = 1.0
             ) -> tf.Tensor:
    """
    Computes the loss for the given data (x_data, y_data) using the training data as input.
    """
    pred = pred_func(x_data, train_data, m, n)
    loss = tf.losses.MSE(y_data, pred[:, 1])
    return loss

In [144]:
# Generate data coordinates using tf.range
x_train = tf.range(-3, 4, 1, dtype=tf.float64)
x_val   = tf.range(-2.5, 3.5, 1, dtype=tf.float64)
x_true  = tf.range(-3.0, 4.0, 0.5, dtype=tf.float64)

# Generate target values through the statistical model
tf.random.set_seed(100)
y_train = y_stat(x_train)
y_val   = y_stat(x_val)
y_true  = y(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = tf.concat([tf.expand_dims(x_train, axis=1), tf.expand_dims(y_train, axis=1)], axis=1)
val_data   = tf.concat([tf.expand_dims(x_val, axis=1), tf.expand_dims(y_val, axis=1)], axis=1)
true_data  = tf.concat([tf.expand_dims(x_true, axis=1), tf.expand_dims(y_true, axis=1)], axis=1)
train_data

# Notice that the instance of data generated by TensorFlow2 is different than the one generated by numpy. To reproduce the same results as with numpy, one would need to generate data with numpy and then convert it to TensorFlow2 tensors. Comment/uncomment the following lines to use different/same data

# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 4., 1.)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat_numpy(x_train)
y_val = y_stat_numpy(x_val)
y_true = y_numpy(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

# Convert numpy arrays to TensorFlow2 tensors
x_train = tf.constant(x_train, dtype=tf.float64)
y_train = tf.constant(y_train, dtype=tf.float64)
x_val = tf.constant(x_val, dtype=tf.float64)
y_val = tf.constant(y_val, dtype=tf.float64)
x_true = tf.constant(x_true, dtype=tf.float64)
y_true = tf.constant(y_true, dtype=tf.float64)
train_data = tf.constant(train_data, dtype=tf.float64)
val_data = tf.constant(val_data, dtype=tf.float64)
true_data = tf.constant(true_data, dtype=tf.float64)
train_data

<tf.Tensor: shape=(7, 2), dtype=float64, numpy=
array([[-3.        ,  2.87523453],
       [-2.        ,  3.00934707],
       [-1.        ,  2.7780358 ],
       [ 0.        ,  1.74756396],
       [ 1.        ,  4.27298745],
       [ 2.        ,  4.51421884],
       [ 3.        ,  1.84617967]])>

In [146]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.89369144]
 [-2.5         2.9422908 ]
 [-2.          2.82663516]
 [-1.5         2.89369144]
 [-1.          2.37845552]
 [-0.5         2.26279988]
 [ 0.          3.52551163]
 [ 0.5         3.01027571]
 [ 1.          3.1308914 ]
 [ 1.5         4.39360315]
 [ 2.          3.05958356]
 [ 2.5         3.18019926]
 [ 3.          4.39360315]]


In [147]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


In [148]:
train_loss = loss_func(x_train, y_train, train_data, 1.0, 1.0)
val_loss = loss_func(x_val, y_val, train_data, 1.0, 1.0)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


#### Optimizing loss function over train data

In [ ]:
# Convert m and n into tf.Variable, so they can be updated by the optimizer.
m = tf.Variable(1.0, dtype=tf.float64)
n = tf.Variable(1.0, dtype=tf.float64)

# Using the Adam optimizer
optimizer = tf.optimizers.Adam(learning_rate=0.01)

# Number of iterations for the optimization loop
num_iterations = 2000

# Training loop: optimize the loss on the training data over m and n.
for i in range(num_iterations):
    with tf.GradientTape() as tape:
        # Compute the loss on the training data for current m and n.
        loss_val = loss_func(x_train, y_train, train_data, m, n)
    # Compute gradients of the loss with respect to m and n.
    grads = tape.gradient(loss_val, [m, n])
    # Update m and n using the optimizer.
    optimizer.apply_gradients(zip(grads, [m, n]))
    
    # Optionally print progress every 100 iterations.
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss_val.numpy():.6f}, m = {m.numpy():.6f}, n = {n.numpy():.6f}")

# After training, evaluate the optimized training and validation loss.
train_loss_opt = loss_func(x_train, y_train, train_data, m, n)
val_loss_opt = loss_func(x_val, y_val, train_data, m, n)

print("Optimal parameters on training set:", m.numpy(), n.numpy())
print("Train loss:", train_loss_opt.numpy())
print("Validation loss:", val_loss_opt.numpy())

Iteration 0: Loss = 1.894886, m = 1.010000, n = 0.990000
Iteration 100: Loss = 1.662379, m = 0.966636, n = 0.833183
Iteration 200: Loss = 1.478976, m = 0.672585, n = 0.416090
Iteration 300: Loss = 0.681452, m = 0.242611, n = -0.239853
Iteration 400: Loss = 0.681307, m = 0.243944, n = -0.237430
Iteration 500: Loss = 0.681307, m = 0.243957, n = -0.237407
Iteration 600: Loss = 0.681307, m = 0.243957, n = -0.237407
Iteration 700: Loss = 0.681307, m = 0.243956, n = -0.237407
Iteration 800: Loss = 0.681308, m = 0.244024, n = -0.237445
Iteration 900: Loss = 0.681307, m = 0.243956, n = -0.237407
Optimal parameters on training set: 0.24395653584119117 -0.23740719094297807
Train loss: 0.6813074745247695
Validation loss: 2.123088458309086


#### Optimizing loss function over validation data

In [153]:
# Convert m and n into tf.Variable, so they can be updated by the optimizer.
m = tf.Variable(1.0, dtype=tf.float64)
n = tf.Variable(1.0, dtype=tf.float64)

# Using the Adam optimizer
optimizer = tf.optimizers.Adam(learning_rate=0.01)

# Number of iterations for the optimization loop
num_iterations = 2000

# Training loop: optimize the loss on the training data over m and n.
for i in range(num_iterations):
    with tf.GradientTape() as tape:
        # Compute the loss on the training data for current m and n.
        loss_val = loss_func(x_val, y_val, train_data, m, n)
    # Compute gradients of the loss with respect to m and n.
    grads = tape.gradient(loss_val, [m, n])
    # Update m and n using the optimizer.
    optimizer.apply_gradients(zip(grads, [m, n]))
    
    # Optionally print progress every 100 iterations.
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss_val.numpy():.6f}, m = {m.numpy():.6f}, n = {n.numpy():.6f}")

# After training, evaluate the optimized training and validation loss.
train_loss_opt = loss_func(x_train, y_train, train_data, m, n)
val_loss_opt = loss_func(x_val, y_val, train_data, m, n)

print("Optimal parameters on training set:", m.numpy(), n.numpy())
print("Train loss:", train_loss_opt.numpy())
print("Validation loss:", val_loss_opt.numpy())

Iteration 0: Loss = 0.404374, m = 1.010000, n = 0.990000
Iteration 100: Loss = 0.184081, m = 1.100149, n = 1.013458
Iteration 200: Loss = 0.176540, m = 1.167979, n = 1.108770
Iteration 300: Loss = 0.170552, m = 1.237681, n = 1.206238
Iteration 400: Loss = 0.166286, m = 1.302654, n = 1.296596
Iteration 500: Loss = 0.163367, m = 1.360773, n = 1.377039
Iteration 600: Loss = 0.161406, m = 1.411631, n = 1.447145
Iteration 700: Loss = 0.160105, m = 1.455489, n = 1.507397
Iteration 800: Loss = 0.159252, m = 1.492872, n = 1.558606
Iteration 900: Loss = 0.158699, m = 1.524391, n = 1.601682
Iteration 1000: Loss = 0.158347, m = 1.550678, n = 1.637538
Iteration 1100: Loss = 0.158126, m = 1.572350, n = 1.667052
Iteration 1200: Loss = 0.157990, m = 1.589996, n = 1.691052
Iteration 1300: Loss = 0.157909, m = 1.604170, n = 1.710310
Iteration 1400: Loss = 0.157861, m = 1.615387, n = 1.725539
Iteration 1500: Loss = 0.157834, m = 1.624121, n = 1.737390
Iteration 1600: Loss = 0.157819, m = 1.630805, n = 1

### Generalized 2NN (polynomial data) - Large Sample

In [155]:
def y(x: tf.Tensor) -> tf.Tensor:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A tf tensor of x values.
    
    Returns:
        A tf tensor of y values computed from the polynomial expression.
    """
    return tf.cast(-tf.pow(x, 4)/24 - tf.pow(x, 3)/6 + tf.pow(x, 2)/2 + x + 2, dtype=tf.float64)
    
def y_stat(x: tf.Tensor) -> tf.Tensor:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A tf tensor of x values.
    
    Returns:
        A tf tensor of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: tf.Tensor = y(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: tf.Tensor = true_y_values + tf.random.normal(tf.shape(true_y_values), mean=0.0, stddev=0.1, dtype=tf.float64)
    
    return noisy_y_values

def y_numpy(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat_numpy(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y_numpy(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 0.1)
    
    return noisy_y_values

def pred_func(x_data: tf.Tensor, 
              train_data: tf.Tensor,
              m: float = 1.0,
              n: float = 1.0
             ) -> tf.Tensor:
    """
    Computes the predictions for the given x_data using a k-nearest neighbors approach 
    based on the training data. If x_data is the same as x_train, it uses the 2nd and 3rd 
    nearest neighbors (to avoid self-match); otherwise, it uses the two nearest neighbors.
    
    Args:
        x_data (tf.Tensor): Tensor of input x values for which to predict y.
        train_data (tf.Tensor): Tensor of training data with shape (N, 2), where 
                                train_data[:, 0] contains the x values and 
                                train_data[:, 1] contains the corresponding y values.
        m (float): The m parameter of the modified distance metric.
        n (float): The n parameter of the modified distance metric.              
    
    Returns:
        tf.Tensor: A tensor of predictions, each row is [x, pred_y].
    """
    x_train = train_data[:, 0]
    y_train = train_data[:, 1]
    
    # If x_data has the same shape as x_train and any element is equal, assume we're predicting on training data.
    if x_data.shape == x_train.shape and tf.reduce_any(tf.equal(x_data, x_train)):
        input_tensor = x_train
    else:
        input_tensor = x_data

    def compute_pred_for_x(x):
        # Compute absolute differences between this x and all training x's.
        diffs = tf.abs(x_train - x)
        # Sort indices of training points by increasing difference.
        sorted_indices = tf.argsort(diffs, direction='ASCENDING')
        # If x is in the training set, skip the closest (self-match) and take the 2nd and 3rd nearest.
        # Otherwise, take the two nearest neighbors.
        if tf.reduce_any(tf.equal(x, x_train)):
            neighbor_indices = sorted_indices[1:3]
        else:
            neighbor_indices = sorted_indices[0:2]
        gathered = tf.gather(y_train, neighbor_indices)
        pred_y = 1/2*tf.reduce_sum(tf.abs(gathered)**n)**(1/m)
        return tf.stack([x, pred_y])
    
    pred = tf.map_fn(compute_pred_for_x, input_tensor, dtype=tf.float64)
    return pred

def loss_func(x_data: tf.Tensor,
              y_data: tf.Tensor,
              train_data: tf.Tensor,
              m: float = 1.0,
              n: float = 1.0
             ) -> tf.Tensor:
    """
    Computes the loss for the given data (x_data, y_data) using the training data as input.
    """
    pred = pred_func(x_data, train_data, m, n)
    loss = tf.losses.MSE(y_data, pred[:, 1])
    return loss

In [158]:
# Generate data coordinates using tf.range
x_train = tf.range(-3, 3.1, 0.1, dtype=tf.float64)
x_val   = tf.range(-2.95, 2.75, 0.2, dtype=tf.float64)
x_test  = tf.range(-2.85, 2.85, 0.2, dtype=tf.float64)
x_true  = tf.range(-3.0, 3.1, 0.5, dtype=tf.float64)

# Generate target values through the statistical model
tf.random.set_seed(100)
y_train = y_stat(x_train)
y_val   = y_stat(x_val)
y_test  = y_stat(x_test)
y_true  = y(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = tf.concat([tf.expand_dims(x_train, axis=1), tf.expand_dims(y_train, axis=1)], axis=1)
val_data   = tf.concat([tf.expand_dims(x_val, axis=1), tf.expand_dims(y_val, axis=1)], axis=1)
test_data  = tf.concat([tf.expand_dims(x_test, axis=1), tf.expand_dims(y_test, axis=1)], axis=1)
true_data  = tf.concat([tf.expand_dims(x_true, axis=1), tf.expand_dims(y_true, axis=1)], axis=1)
train_data

# Notice that the instance of data generated by TensorFlow2 is different than the one generated by numpy. To reproduce the same results as with numpy, one would need to generate data with numpy and then convert it to TensorFlow2 tensors. Comment/uncomment the following lines to use different/same data

# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 3.1, 0.1)
x_val: np.ndarray = np.arange(-2.95, 2.75, 0.2)
x_test: np.ndarray = np.arange(-2.85, 2.85, 0.2)
x_true: np.ndarray = np.arange(-3., 3.1, 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat_numpy(x_train)
y_val = y_stat_numpy(x_val)
y_test = y_stat_numpy(x_test)
y_true = y_numpy(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
test_data = np.concatenate((x_test.reshape(-1, 1), y_test.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

# Convert numpy arrays to TensorFlow2 tensors
x_train = tf.constant(x_train, dtype=tf.float64)
y_train = tf.constant(y_train, dtype=tf.float64)
x_val = tf.constant(x_val, dtype=tf.float64)
y_val = tf.constant(y_val, dtype=tf.float64)
x_test = tf.constant(x_test, dtype=tf.float64)
y_test = tf.constant(y_test, dtype=tf.float64)
x_true = tf.constant(x_true, dtype=tf.float64)
y_true = tf.constant(y_true, dtype=tf.float64)
train_data = tf.constant(train_data, dtype=tf.float64)
val_data = tf.constant(val_data, dtype=tf.float64)
true_data = tf.constant(true_data, dtype=tf.float64)

In [159]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]

In [160]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 0.021, Validation loss: 0.011


In [161]:
train_loss = loss_func(x_train, y_train, train_data, 1.0, 1.0)
val_loss = loss_func(x_val, y_val, train_data, 1.0, 1.0)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 0.021, Validation loss: 0.011


#### Optimizing loss function over train data

In [163]:
# Convert m and n into tf.Variable, so they can be updated by the optimizer.
m = tf.Variable(1.0, dtype=tf.float64)
n = tf.Variable(1.0, dtype=tf.float64)

# Using the Adam optimizer
optimizer = tf.optimizers.Adam(learning_rate=0.01)

# Number of iterations for the optimization loop
num_iterations = 500

# Training loop: optimize the loss on the training data over m and n.
for i in range(num_iterations):
    with tf.GradientTape() as tape:
        # Compute the loss on the training data for current m and n.
        loss_val = loss_func(x_train, y_train, train_data, m, n)
    # Compute gradients of the loss with respect to m and n.
    grads = tape.gradient(loss_val, [m, n])
    # Update m and n using the optimizer.
    optimizer.apply_gradients(zip(grads, [m, n]))
    
    # Optionally print progress every 100 iterations.
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss_val.numpy():.6f}, m = {m.numpy():.6f}, n = {n.numpy():.6f}")

# After training, evaluate the optimized training and validation loss.
train_loss_opt = loss_func(x_train, y_train, train_data, m, n)
val_loss_opt = loss_func(x_val, y_val, train_data, m, n)

print("Optimal parameters on training set:", m.numpy(), n.numpy())
print("Train loss:", train_loss_opt.numpy())
print("Validation loss:", val_loss_opt.numpy())

Iteration 0: Loss = 0.020867, m = 1.009994, n = 1.009994
Iteration 100: Loss = 0.020650, m = 1.025664, n = 1.040276
Iteration 200: Loss = 0.020634, m = 1.033806, n = 1.053028
Iteration 300: Loss = 0.020634, m = 1.035035, n = 1.054954
Iteration 400: Loss = 0.020634, m = 1.035107, n = 1.055066
Optimal parameters on training set: 1.0351082371491942 1.0550684217315982
Train loss: 0.020633723075049924
Validation loss: 0.010612380061573555


#### Optimizing loss function over validation data

In [164]:
# Convert m and n into tf.Variable, so they can be updated by the optimizer.
m = tf.Variable(1.0, dtype=tf.float64)
n = tf.Variable(1.0, dtype=tf.float64)

# Using the Adam optimizer
optimizer = tf.optimizers.Adam(learning_rate=0.01)

# Number of iterations for the optimization loop
num_iterations = 500

# Training loop: optimize the loss on the training data over m and n.
for i in range(num_iterations):
    with tf.GradientTape() as tape:
        # Compute the loss on the training data for current m and n.
        loss_val = loss_func(x_val, y_val, train_data, m, n)
    # Compute gradients of the loss with respect to m and n.
    grads = tape.gradient(loss_val, [m, n])
    # Update m and n using the optimizer.
    optimizer.apply_gradients(zip(grads, [m, n]))
    
    # Optionally print progress every 100 iterations.
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss_val.numpy():.6f}, m = {m.numpy():.6f}, n = {n.numpy():.6f}")

# After training, evaluate the optimized training and validation loss.
train_loss_opt = loss_func(x_train, y_train, train_data, m, n)
val_loss_opt = loss_func(x_val, y_val, train_data, m, n)

print("Optimal parameters on training set:", m.numpy(), n.numpy())
print("Train loss:", train_loss_opt.numpy())
print("Validation loss:", val_loss_opt.numpy())

Iteration 0: Loss = 0.011172, m = 0.990000, n = 1.010000
Iteration 100: Loss = 0.010636, m = 1.024352, n = 1.040677
Iteration 200: Loss = 0.010460, m = 1.045189, n = 1.073427
Iteration 300: Loss = 0.010432, m = 1.054560, n = 1.088095
Iteration 400: Loss = 0.010430, m = 1.057576, n = 1.092814
Optimal parameters on training set: 1.0582889658411787 1.0939295518102143
Train loss: 0.02080357980944779
Validation loss: 0.01042991889900298


## PyTorch

### 2NN (polynomial data)

In [168]:
def y(x: torch.Tensor) -> torch.Tensor:
    """Generates polynomial data based on a given polynomial expression.

    Args:
        x: A torch tensor of x values.

    Returns:
        A torch tensor of y values computed from the polynomial expression.
    """
    return (-torch.pow(x, 4) / 24 - torch.pow(x, 3) / 6 + torch.pow(x, 2) / 2 + x + 2).double()

def y_stat(x: torch.Tensor) -> torch.Tensor:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.

    Args:
        x: A torch tensor of x values.

    Returns:
        A torch tensor of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values = y(x)

    # Add Gaussian noise with mean 0 and standard deviation 1
    noisy_y_values = torch.normal(mean=true_y_values, std=1.0)

    return noisy_y_values

def y_numpy(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat_numpy(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y_numpy(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

def pred_func(x_data: torch.Tensor, 
              train_data: torch.Tensor
             ) -> torch.Tensor:
    """
    Computes the predictions for the given x_data using a k-nearest neighbors approach 
    based on the training data. If x_data is the same as x_train, it uses the 2nd and 3rd 
    nearest neighbors (to avoid self-match); otherwise, it uses the two nearest neighbors.
    
    Args:
        x_data (torch.Tensor): Tensor of input x values for which to predict y.
        train_data (torch.Tensor): Tensor of training data with shape (N, 2), where 
            train_data[:, 0] contains the x values and train_data[:, 1] contains the corresponding y values.
    
    Returns:
        torch.Tensor: A tensor of predictions, each row is [x, pred_y].
    """
    x_train = train_data[:, 0]
    y_train = train_data[:, 1]
    
    # If x_data has the same shape as x_train and any element equals one in x_train,
    # assume we are predicting on training data.
    if x_data.shape == x_train.shape and torch.any(x_data == x_train):
        input_tensor = x_train
    else:
        input_tensor = x_data

    def compute_pred_for_x(x):
        # Compute absolute differences between this x and all training x's.
        diffs = torch.abs(x_train - x)
        # Sort indices by increasing difference.
        sorted_indices = torch.argsort(diffs, descending=False)
        # If x is in the training set, skip the closest (self-match) and use 2nd and 3rd nearest neighbors.
        if torch.any(x == x_train):
            neighbor_indices = sorted_indices[1:3]
        else:
            neighbor_indices = sorted_indices[0:2]
        gathered = y_train[neighbor_indices]
        pred_y = torch.mean(gathered)
        return torch.stack([x, pred_y])
    
    # Compute predictions for each x in input_tensor using a list comprehension.
    preds = [compute_pred_for_x(x) for x in input_tensor]
    pred = torch.stack(preds)
    return pred

def MSE(y_true: torch.Tensor, 
        y_pred: torch.Tensor
       ) -> torch.Tensor:
    """
    Calculates the Mean Squared Error (MSE) using PyTorch's MSE loss function.
    
    Args:
        x_values: A 1D torch.Tensor of x values for evaluation.
        y_true: A 1D torch.Tensor of true y values.
    
    Returns:
        The MSE as a torch.Tensor.
    """
    loss: torch.Tensor = F.mse_loss(y_true, y_pred)
    return loss

In [170]:
# Generate data coordinates using tf.range
x_train: torch.Tensor = torch.arange(-3., 4., 1, dtype=torch.float64)
x_val: torch.Tensor = torch.arange(-2.5, 3.5, 1, dtype=torch.float64)
x_true: torch.Tensor = torch.arange(-3., 4., 0.5, dtype=torch.float64)

# Generate target values with noise
torch.manual_seed(100)
y_train: torch.Tensor = y_stat(x_train)
y_val: torch.Tensor = y_stat(x_val)
y_true: torch.Tensor = y(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = torch.stack((x_train, y_train), dim=1)
val_data   = torch.stack((x_val, y_val), dim=1)
true_data  = torch.stack((x_true, y_true), dim=1)
train_data

# Notice that the instance of data generated by TensorFlow2 is different than the one generated by numpy. To reproduce the same results as with numpy, one would need to generate data with numpy and then convert it to TensorFlow2 tensors. Comment/uncomment the following lines to use different/same data

# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 4., 1.)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat_numpy(x_train)
y_val = y_stat_numpy(x_val)
y_true = y_numpy(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

# Convert numpy arrays to TensorFlow2 tensors
x_train = torch.tensor(x_train, dtype=torch.float64)
y_train = torch.tensor(y_train, dtype=torch.float64)
x_val = torch.tensor(x_val, dtype=torch.float64)
y_val = torch.tensor(y_val, dtype=torch.float64)
x_true = torch.tensor(x_true, dtype=torch.float64)
y_true = torch.tensor(y_true, dtype=torch.float64)
train_data = torch.stack((x_train, y_train), dim=1)
val_data = torch.stack((x_val, y_val), dim=1)
true_data = torch.stack((x_true, y_true), dim=1)
train_data

tensor([[-3.0000,  2.8752],
        [-2.0000,  3.0093],
        [-1.0000,  2.7780],
        [ 0.0000,  1.7476],
        [ 1.0000,  4.2730],
        [ 2.0000,  4.5142],
        [ 3.0000,  1.8462]], dtype=torch.float64)

In [171]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.89369144]
 [-2.5         2.9422908 ]
 [-2.          2.82663516]
 [-1.5         2.89369144]
 [-1.          2.37845552]
 [-0.5         2.26279988]
 [ 0.          3.52551163]
 [ 0.5         3.01027571]
 [ 1.          3.1308914 ]
 [ 1.5         4.39360315]
 [ 2.          3.05958356]
 [ 2.5         3.18019926]
 [ 3.          4.39360315]]


In [172]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


### 2NN (non-polynomial data)

In [173]:
def y(x: torch.Tensor) -> torch.Tensor:
    """Computes the function 3 + sin(x) - cos(x) for given x values.
    
    Args:
        x: A torch tensor of x values.
    
    Returns:
        A torch tensor of y values computed from the function.
    """
    return (3 + torch.sin(x) - torch.cos(x)).double()

def y_stat(x: torch.Tensor) -> torch.Tensor:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.

    Args:
        x: A torch tensor of x values.

    Returns:
        A torch tensor of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values = y(x)

    # Add Gaussian noise with mean 0 and standard deviation 1
    noisy_y_values = torch.normal(mean=true_y_values, std=1.0)

    return noisy_y_values

def y_numpy(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return 3 + np.sin(x) - np.cos(x)

def y_stat_numpy(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y_numpy(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

In [174]:
# Generate data coordinates using tf.range
x_train: torch.Tensor = torch.arange(-3., 4., 1, dtype=torch.float64)
x_val: torch.Tensor = torch.arange(-2.5, 3.5, 1, dtype=torch.float64)
x_true: torch.Tensor = torch.arange(-3., 4., 0.5, dtype=torch.float64)

# Generate target values with noise
torch.manual_seed(100)
y_train: torch.Tensor = y_stat(x_train)
y_val: torch.Tensor = y_stat(x_val)
y_true: torch.Tensor = y(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = torch.stack((x_train, y_train), dim=1)
val_data   = torch.stack((x_val, y_val), dim=1)
true_data  = torch.stack((x_true, y_true), dim=1)
train_data

# Notice that the instance of data generated by TensorFlow2 is different than the one generated by numpy. To reproduce the same results as with numpy, one would need to generate data with numpy and then convert it to TensorFlow2 tensors. Comment/uncomment the following lines to use different/same data

# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 4., 1.)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat_numpy(x_train)
y_val = y_stat_numpy(x_val)
y_true = y_numpy(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

# Convert numpy arrays to TensorFlow2 tensors
x_train = torch.tensor(x_train, dtype=torch.float64)
y_train = torch.tensor(y_train, dtype=torch.float64)
x_val = torch.tensor(x_val, dtype=torch.float64)
y_val = torch.tensor(y_val, dtype=torch.float64)
x_true = torch.tensor(x_true, dtype=torch.float64)
y_true = torch.tensor(y_true, dtype=torch.float64)
train_data = torch.stack((x_train, y_train), dim=1)
val_data = torch.stack((x_val, y_val), dim=1)
true_data = torch.stack((x_true, y_true), dim=1)
train_data

tensor([[-3.0000,  2.0991],
        [-2.0000,  2.8495],
        [-1.0000,  2.7713],
        [ 0.0000,  1.7476],
        [ 1.0000,  4.2825],
        [ 2.0000,  4.8397],
        [ 3.0000,  4.3523]], dtype=torch.float64)

In [175]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.81039616]
 [-2.5         2.47431841]
 [-2.          2.43518476]
 [-1.5         2.81039616]
 [-1.          2.29854689]
 [-0.5         2.25941324]
 [ 0.          3.52687599]
 [ 0.5         3.01502671]
 [ 1.          3.29361353]
 [ 1.5         4.56107629]
 [ 2.          4.31739082]
 [ 2.5         4.59597764]
 [ 3.          4.56107629]]


In [176]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 0.766, Validation loss: 0.466


### Generalized 2NN (polynomial data)

In [187]:
def y(x: torch.Tensor) -> torch.Tensor:
    """Generates polynomial data based on a given polynomial expression.

    Args:
        x: A torch tensor of x values.

    Returns:
        A torch tensor of y values computed from the polynomial expression.
    """
    return (-torch.pow(x, 4) / 24 - torch.pow(x, 3) / 6 + torch.pow(x, 2) / 2 + x + 2).double()

def y_stat(x: torch.Tensor) -> torch.Tensor:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.

    Args:
        x: A torch tensor of x values.

    Returns:
        A torch tensor of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values = y(x)

    # Add Gaussian noise with mean 0 and standard deviation 1
    noisy_y_values = torch.normal(mean=true_y_values, std=1.0)

    return noisy_y_values

def y_numpy(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat_numpy(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y_numpy(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 1)
    
    return noisy_y_values

def pred_func(x_data: torch.Tensor, 
              train_data: torch.Tensor,
              m: float = 1.0,
              n: float = 1.0
             ) -> torch.Tensor:
    """
    Computes the predictions for the given x_data using a k-nearest neighbors approach 
    based on the training data. If x_data is the same as x_train, it uses the 2nd and 3rd 
    nearest neighbors (to avoid self-match); otherwise, it uses the two nearest neighbors.
    
    Args:
        x_data (tf.Tensor): Tensor of input x values for which to predict y.
        train_data (tf.Tensor): Tensor of training data with shape (N, 2), where 
                                train_data[:, 0] contains the x values and 
                                train_data[:, 1] contains the corresponding y values.
        m (float): The m parameter of the modified distance metric.
        n (float): The n parameter of the modified distance metric.              
    
    Returns:
        tf.Tensor: A tensor of predictions, each row is [x, pred_y].
    """
    x_train = train_data[:, 0]
    y_train = train_data[:, 1]
    
    # If x_data has the same shape as x_train and any element equals one in x_train,
    # assume we are predicting on training data.
    if x_data.shape == x_train.shape and torch.any(x_data == x_train):
        input_tensor = x_train
    else:
        input_tensor = x_data

    def compute_pred_for_x(x):
        # Compute absolute differences between this x and all training x's.
        diffs = torch.abs(x_train - x)
        # Sort indices by increasing difference.
        sorted_indices = torch.argsort(diffs, descending=False)
        # If x is in the training set, skip the closest (self-match) and use 2nd and 3rd nearest neighbors.
        if torch.any(x == x_train):
            neighbor_indices = sorted_indices[1:3]
        else:
            neighbor_indices = sorted_indices[0:2]
        gathered = y_train[neighbor_indices]
        pred_y = 1/2*torch.sum(torch.abs(gathered)**n)**(1/m)
        return torch.stack([x, pred_y])
    
    # Compute predictions for each x in input_tensor using a list comprehension.
    preds = [compute_pred_for_x(x) for x in input_tensor]
    pred = torch.stack(preds)
    return pred

def loss_func(x_data: torch.Tensor,
              y_data: torch.Tensor,
              train_data: torch.Tensor,
              m: float = 1.0,
              n: float = 1.0
             ) -> torch.Tensor:
    """
    Computes the loss for the given data (x_data, y_data) using the training data as input.
    """
    pred: torch.Tensor = pred_func(x_data, train_data, m, n)
    loss: torch.Tensor = F.mse_loss(y_data, pred[:, 1])
    return loss

In [188]:
# Generate data coordinates using tf.range
x_train: torch.Tensor = torch.arange(-3., 4., 1, dtype=torch.float64)
x_val: torch.Tensor = torch.arange(-2.5, 3.5, 1, dtype=torch.float64)
x_true: torch.Tensor = torch.arange(-3., 4., 0.5, dtype=torch.float64)

# Generate target values with noise
torch.manual_seed(100)
y_train: torch.Tensor = y_stat(x_train)
y_val: torch.Tensor = y_stat(x_val)
y_true: torch.Tensor = y(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = torch.stack((x_train, y_train), dim=1)
val_data   = torch.stack((x_val, y_val), dim=1)
true_data  = torch.stack((x_true, y_true), dim=1)
train_data

# Notice that the instance of data generated by TensorFlow2 is different than the one generated by numpy. To reproduce the same results as with numpy, one would need to generate data with numpy and then convert it to TensorFlow2 tensors. Comment/uncomment the following lines to use different/same data

# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 4., 1.)
x_val: np.ndarray = np.arange(-2.5, 3.5, 1)
x_true: np.ndarray = np.arange(-3., 4., 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat_numpy(x_train)
y_val = y_stat_numpy(x_val)
y_true = y_numpy(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

# Convert numpy arrays to TensorFlow2 tensors
x_train = torch.tensor(x_train, dtype=torch.float64)
y_train = torch.tensor(y_train, dtype=torch.float64)
x_val = torch.tensor(x_val, dtype=torch.float64)
y_val = torch.tensor(y_val, dtype=torch.float64)
x_true = torch.tensor(x_true, dtype=torch.float64)
y_true = torch.tensor(y_true, dtype=torch.float64)
train_data = torch.stack((x_train, y_train), dim=1)
val_data = torch.stack((x_val, y_val), dim=1)
true_data = torch.stack((x_true, y_true), dim=1)
train_data

tensor([[-3.0000,  2.8752],
        [-2.0000,  3.0093],
        [-1.0000,  2.7780],
        [ 0.0000,  1.7476],
        [ 1.0000,  4.2730],
        [ 2.0000,  4.5142],
        [ 3.0000,  1.8462]], dtype=torch.float64)

In [189]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]
print(sorted_a)

[[-3.          2.89369144]
 [-2.5         2.9422908 ]
 [-2.          2.82663516]
 [-1.5         2.89369144]
 [-1.          2.37845552]
 [-0.5         2.26279988]
 [ 0.          3.52551163]
 [ 0.5         3.01027571]
 [ 1.          3.1308914 ]
 [ 1.5         4.39360315]
 [ 2.          3.05958356]
 [ 2.5         3.18019926]
 [ 3.          4.39360315]]


In [190]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


In [191]:
train_loss = loss_func(x_train, y_train, train_data, 1.0, 1.0)
val_loss = loss_func(x_val, y_val, train_data, 1.0, 1.0)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 1.895, Validation loss: 0.404


#### Optimizing loss function over train data

In [193]:
# Create parameters m and n as torch tensors with gradients enabled.
m = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
n = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)

# Use the Adam optimizer with a learning rate of 0.01
optimizer = torch.optim.Adam([m, n], lr=0.01)

# Number of iterations for the optimization loop
num_iterations = 1000

# Training loop: optimize the loss on the training data over m and n.
for i in range(num_iterations):
    optimizer.zero_grad()                      # Clear gradients from the previous step.
    loss_val = loss_func(x_train, y_train, train_data, m, n)
    loss_val.backward()                        # Compute gradients.
    optimizer.step()                           # Update m and n.

    # Optionally print progress every 100 iterations.
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss_val.item():.6f}, m = {m.item():.6f}, n = {n.item():.6f}")

# After training, evaluate the optimized training and validation loss.
train_loss_opt = loss_func(x_train, y_train, train_data, m, n)
val_loss_opt   = loss_func(x_val, y_val, train_data, m, n)

print("Optimal parameters on training set:", m.item(), n.item())
print("Train loss:", train_loss_opt.item())
print("Validation loss:", val_loss_opt.item())

Iteration 0: Loss = 1.894886, m = 1.010000, n = 0.990000
Iteration 100: Loss = 1.662379, m = 0.966635, n = 0.833182
Iteration 200: Loss = 1.478971, m = 0.672580, n = 0.416082
Iteration 300: Loss = 0.681452, m = 0.242612, n = -0.239852
Iteration 400: Loss = 0.681307, m = 0.243944, n = -0.237430
Iteration 500: Loss = 0.681307, m = 0.243957, n = -0.237407
Iteration 600: Loss = 0.681307, m = 0.243957, n = -0.237407
Iteration 700: Loss = 0.681307, m = 0.243956, n = -0.237407
Iteration 800: Loss = 0.681311, m = 0.243908, n = -0.237349
Iteration 900: Loss = 0.681307, m = 0.243957, n = -0.237407
Optimal parameters on training set: 0.24395653875897472 -0.2374071936945846
Train loss: 0.6813074745247755
Validation loss: 2.1230883033003125


#### Optimizing loss function over validation data

In [194]:
# Create parameters m and n as torch tensors with gradients enabled.
m = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
n = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)

# Use the Adam optimizer with a learning rate of 0.01
optimizer = torch.optim.Adam([m, n], lr=0.01)

# Number of iterations for the optimization loop
num_iterations = 1000

# Training loop: optimize the loss on the training data over m and n.
for i in range(num_iterations):
    optimizer.zero_grad()                      # Clear gradients from the previous step.
    loss_val = loss_func(x_val, y_val, train_data, m, n)
    loss_val.backward()                        # Compute gradients.
    optimizer.step()                           # Update m and n.

    # Optionally print progress every 100 iterations.
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss_val.item():.6f}, m = {m.item():.6f}, n = {n.item():.6f}")

# After training, evaluate the optimized training and validation loss.
train_loss_opt = loss_func(x_train, y_train, train_data, m, n)
val_loss_opt   = loss_func(x_val, y_val, train_data, m, n)

print("Optimal parameters on training set:", m.item(), n.item())
print("Train loss:", train_loss_opt.item())
print("Validation loss:", val_loss_opt.item())

Iteration 0: Loss = 0.404374, m = 1.010000, n = 0.990000
Iteration 100: Loss = 0.184081, m = 1.100149, n = 1.013459
Iteration 200: Loss = 0.176540, m = 1.167979, n = 1.108771
Iteration 300: Loss = 0.170552, m = 1.237682, n = 1.206240
Iteration 400: Loss = 0.166286, m = 1.302655, n = 1.296598
Iteration 500: Loss = 0.163367, m = 1.360775, n = 1.377040
Iteration 600: Loss = 0.161406, m = 1.411632, n = 1.447147
Iteration 700: Loss = 0.160105, m = 1.455491, n = 1.507399
Iteration 800: Loss = 0.159252, m = 1.492873, n = 1.558608
Iteration 900: Loss = 0.158699, m = 1.524392, n = 1.601684
Optimal parameters on training set: 1.5504399708712864 1.637213436488031
Train loss: 1.7149927387267232
Validation loss: 0.1583467068696484


### Generalized 2NN (polynomial data) - Large Sample

In [195]:
def y(x: torch.Tensor) -> torch.Tensor:
    """Generates polynomial data based on a given polynomial expression.

    Args:
        x: A torch tensor of x values.

    Returns:
        A torch tensor of y values computed from the polynomial expression.
    """
    return (-torch.pow(x, 4) / 24 - torch.pow(x, 3) / 6 + torch.pow(x, 2) / 2 + x + 2).double()

def y_stat(x: torch.Tensor) -> torch.Tensor:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.

    Args:
        x: A torch tensor of x values.

    Returns:
        A torch tensor of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values = y(x)

    # Add Gaussian noise with mean 0 and standard deviation 1
    noisy_y_values = torch.normal(mean=true_y_values, std=0.1)

    return noisy_y_values

def y_numpy(x: np.ndarray) -> np.ndarray:
    """Generates polynomial data based on a given polynomial expression.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression.
    """
    return -x**4/24 - x**3/6 + x**2/2 + x + 2

def y_stat_numpy(x: np.ndarray) -> np.ndarray:
    """Generates noisy polynomial data by adding Gaussian noise to the true y values.
    
    Args:
        x: A numpy array of x values.
    
    Returns:
        A numpy array of y values computed from the polynomial expression with added Gaussian noise.
    """
    # Generate true y values based on a polynomial expression
    true_y_values: np.ndarray = y_numpy(x)
    
    # Add Gaussian noise with a mean of 0 and standard deviation of 1 to the true y values
    noisy_y_values: np.ndarray = np.random.normal(true_y_values, 0.1)
    
    return noisy_y_values

def pred_func(x_data: torch.Tensor, 
              train_data: torch.Tensor,
              m: float = 1.0,
              n: float = 1.0
             ) -> torch.Tensor:
    """
    Computes the predictions for the given x_data using a k-nearest neighbors approach 
    based on the training data. If x_data is the same as x_train, it uses the 2nd and 3rd 
    nearest neighbors (to avoid self-match); otherwise, it uses the two nearest neighbors.
    
    Args:
        x_data (tf.Tensor): Tensor of input x values for which to predict y.
        train_data (tf.Tensor): Tensor of training data with shape (N, 2), where 
                                train_data[:, 0] contains the x values and 
                                train_data[:, 1] contains the corresponding y values.
        m (float): The m parameter of the modified distance metric.
        n (float): The n parameter of the modified distance metric.              
    
    Returns:
        tf.Tensor: A tensor of predictions, each row is [x, pred_y].
    """
    x_train = train_data[:, 0]
    y_train = train_data[:, 1]
    
    # If x_data has the same shape as x_train and any element equals one in x_train,
    # assume we are predicting on training data.
    if x_data.shape == x_train.shape and torch.any(x_data == x_train):
        input_tensor = x_train
    else:
        input_tensor = x_data

    def compute_pred_for_x(x):
        # Compute absolute differences between this x and all training x's.
        diffs = torch.abs(x_train - x)
        # Sort indices by increasing difference.
        sorted_indices = torch.argsort(diffs, descending=False)
        # If x is in the training set, skip the closest (self-match) and use 2nd and 3rd nearest neighbors.
        if torch.any(x == x_train):
            neighbor_indices = sorted_indices[1:3]
        else:
            neighbor_indices = sorted_indices[0:2]
        gathered = y_train[neighbor_indices]
        pred_y = 1/2*torch.sum(torch.abs(gathered)**n)**(1/m)
        return torch.stack([x, pred_y])
    
    # Compute predictions for each x in input_tensor using a list comprehension.
    preds = [compute_pred_for_x(x) for x in input_tensor]
    pred = torch.stack(preds)
    return pred

def loss_func(x_data: torch.Tensor,
              y_data: torch.Tensor,
              train_data: torch.Tensor,
              m: float = 1.0,
              n: float = 1.0
             ) -> torch.Tensor:
    """
    Computes the loss for the given data (x_data, y_data) using the training data as input.
    """
    pred: torch.Tensor = pred_func(x_data, train_data, m, n)
    loss: torch.Tensor = F.mse_loss(y_data, pred[:, 1])
    return loss

In [196]:
# Generate data coordinates using tf.range
x_train: torch.Tensor = torch.arange(-3, 3.1, 0.1, dtype=torch.float64)
x_val: torch.Tensor = torch.arange(-2.95, 2.75, 0.2, dtype=torch.float64)
x_test: torch.Tensor = torch.arange(-2.85, 2.85, 0.2, dtype=torch.float64)
x_true: torch.Tensor = torch.arange(-3, 3.1, 0.5, dtype=torch.float64)

# Generate target values with noise
torch.manual_seed(100)
y_train: torch.Tensor = y_stat(x_train)
y_val: torch.Tensor = y_stat(x_val)
y_test: torch.Tensor = y_stat(x_test)
y_true: torch.Tensor = y(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = torch.stack((x_train, y_train), dim=1)
val_data   = torch.stack((x_val, y_val), dim=1)
test_data  = torch.stack((x_test, y_test), dim=1)
true_data  = torch.stack((x_true, y_true), dim=1)

# Notice that the instance of data generated by TensorFlow2 is different than the one generated by numpy. To reproduce the same results as with numpy, one would need to generate data with numpy and then convert it to TensorFlow2 tensors. Comment/uncomment the following lines to use different/same data

# Generate data coordinates
x_train: np.ndarray = np.arange(-3., 3.1, 0.1)
x_val: np.ndarray = np.arange(-2.95, 2.75, 0.2)
x_test: np.ndarray = np.arange(-2.85, 2.85, 0.2)
x_true: np.ndarray = np.arange(-3., 3.1, 0.5)

# Generate target values through the statistical model
np.random.seed(100)
y_train = y_stat_numpy(x_train)
y_val = y_stat_numpy(x_val)
y_test = y_stat_numpy(x_test)
y_true = y_numpy(x_true)

# Create data arrays by concatenating x and y values column-wise
train_data = np.concatenate((x_train.reshape(-1, 1), y_train.reshape(-1, 1)), axis=1)
val_data = np.concatenate((x_val.reshape(-1, 1), y_val.reshape(-1, 1)), axis=1)
test_data = np.concatenate((x_test.reshape(-1, 1), y_test.reshape(-1, 1)), axis=1)
true_data = np.concatenate((x_true.reshape(-1, 1), y_true.reshape(-1, 1)), axis=1)

# Convert numpy arrays to TensorFlow2 tensors
x_train = torch.tensor(x_train, dtype=torch.float64)
y_train = torch.tensor(y_train, dtype=torch.float64)
x_val = torch.tensor(x_val, dtype=torch.float64)
y_val = torch.tensor(y_val, dtype=torch.float64)
x_test = torch.tensor(x_test, dtype=torch.float64)
y_test = torch.tensor(y_test, dtype=torch.float64)
x_true = torch.tensor(x_true, dtype=torch.float64)
y_true = torch.tensor(y_true, dtype=torch.float64)
train_data = torch.stack((x_train, y_train), dim=1)
val_data = torch.stack((x_val, y_val), dim=1)
true_data = torch.stack((x_true, y_true), dim=1)

In [198]:
# computing prediction with non-parametrix k-nearest neighbors (using first 2 neighbors)
pred_train = pred_func(x_train, train_data)
pred_val = pred_func(x_val, train_data)
a = np.concatenate((pred_train, pred_val))
sorted_a = a[np.argsort(a[:, 0])]

In [199]:
train_loss = MSE(y_train, pred_train[:, 1])
val_loss = MSE(y_val, pred_val[:, 1])
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 0.021, Validation loss: 0.011


In [200]:
train_loss = loss_func(x_train, y_train, train_data, 1.0, 1.0)
val_loss = loss_func(x_val, y_val, train_data, 1.0, 1.0)
print(f"Train loss: {train_loss:.3f}, Validation loss: {val_loss:.3f}")

Train loss: 0.021, Validation loss: 0.011


#### Optimizing loss function over train data

In [201]:
# Create parameters m and n as torch tensors with gradients enabled.
m = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
n = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)

# Use the Adam optimizer with a learning rate of 0.01
optimizer = torch.optim.Adam([m, n], lr=0.01)

# Number of iterations for the optimization loop
num_iterations = 1000

# Training loop: optimize the loss on the training data over m and n.
for i in range(num_iterations):
    optimizer.zero_grad()                      # Clear gradients from the previous step.
    loss_val = loss_func(x_train, y_train, train_data, m, n)
    loss_val.backward()                        # Compute gradients.
    optimizer.step()                           # Update m and n.

    # Optionally print progress every 100 iterations.
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss_val.item():.6f}, m = {m.item():.6f}, n = {n.item():.6f}")

# After training, evaluate the optimized training and validation loss.
train_loss_opt = loss_func(x_train, y_train, train_data, m, n)
val_loss_opt   = loss_func(x_val, y_val, train_data, m, n)

print("Optimal parameters on training set:", m.item(), n.item())
print("Train loss:", train_loss_opt.item())
print("Validation loss:", val_loss_opt.item())

Iteration 0: Loss = 0.020867, m = 1.010000, n = 1.010000
Iteration 100: Loss = 0.020650, m = 1.025668, n = 1.040283
Iteration 200: Loss = 0.020634, m = 1.033808, n = 1.053030
Iteration 300: Loss = 0.020634, m = 1.035035, n = 1.054954
Iteration 400: Loss = 0.020634, m = 1.035107, n = 1.055066
Iteration 500: Loss = 0.020634, m = 1.035108, n = 1.055068
Iteration 600: Loss = 0.020634, m = 1.035108, n = 1.055068
Iteration 700: Loss = 0.020634, m = 1.035108, n = 1.055068
Iteration 800: Loss = 0.020634, m = 1.035108, n = 1.055068
Iteration 900: Loss = 0.020634, m = 1.035108, n = 1.055068
Optimal parameters on training set: 1.035108235782879 1.0550684195900095
Train loss: 0.02063372307504991
Validation loss: 0.010612380073538024


#### Optimizing loss function over validation data

In [202]:
# Create parameters m and n as torch tensors with gradients enabled.
m = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)
n = torch.tensor(1.0, dtype=torch.float64, requires_grad=True)

# Use the Adam optimizer with a learning rate of 0.01
optimizer = torch.optim.Adam([m, n], lr=0.01)

# Number of iterations for the optimization loop
num_iterations = 1000

# Training loop: optimize the loss on the training data over m and n.
for i in range(num_iterations):
    optimizer.zero_grad()                      # Clear gradients from the previous step.
    loss_val = loss_func(x_val, y_val, train_data, m, n)
    loss_val.backward()                        # Compute gradients.
    optimizer.step()                           # Update m and n.

    # Optionally print progress every 100 iterations.
    if i % 100 == 0:
        print(f"Iteration {i}: Loss = {loss_val.item():.6f}, m = {m.item():.6f}, n = {n.item():.6f}")

# After training, evaluate the optimized training and validation loss.
train_loss_opt = loss_func(x_train, y_train, train_data, m, n)
val_loss_opt   = loss_func(x_val, y_val, train_data, m, n)

print("Optimal parameters on training set:", m.item(), n.item())
print("Train loss:", train_loss_opt.item())
print("Validation loss:", val_loss_opt.item())

Iteration 0: Loss = 0.011172, m = 0.990000, n = 1.010000
Iteration 100: Loss = 0.010636, m = 1.024351, n = 1.040676
Iteration 200: Loss = 0.010460, m = 1.045188, n = 1.073425
Iteration 300: Loss = 0.010432, m = 1.054560, n = 1.088094
Iteration 400: Loss = 0.010430, m = 1.057576, n = 1.092814
Iteration 500: Loss = 0.010430, m = 1.058292, n = 1.093934
Iteration 600: Loss = 0.010430, m = 1.058416, n = 1.094128
Iteration 700: Loss = 0.010430, m = 1.058431, n = 1.094152
Iteration 800: Loss = 0.010430, m = 1.058433, n = 1.094155
Iteration 900: Loss = 0.010430, m = 1.058433, n = 1.094155
Optimal parameters on training set: 1.0584328519676824 1.0941547015304272
Train loss: 0.02080467709993923
Validation loss: 0.010429915552680049
